# Notebook 1: Exploring the Open-Text Dataset

## Purpose
Before building a coding frame, I want to read the data cold, without imposing
categories yet. This notebook is exploration only. No coding of themes happens here.

## Data source
City of Austin, City Cultural Centers Audit Community Survey, Open Response Data.
Survey conducted January 2020 across four cultural centres, audited by the
Austin City Auditor's Office. Full dataset: 862 open-text responses across four
facilities and four survey prompts (facilities, staff, programmes, fees).

For this project I'm using the **facilities feedback** responses only (204 rows),
since mixing responses from different prompts (e.g. "fees" and "staff") would mean
comparing answers to different questions as if they were one theme space.

## Note on domain
This is municipal survey data, not student survey data. I'm using it to practise
manual open-text coding, the same discipline the Big Conversation project needs,
on a dataset I could access publicly. The subject matter (cultural centre facilities)
doesn't transfer, but the method does.

## What this notebook does
1. Load the raw data
2. Filter to the facilities-feedback responses
3. Check basic structure: response lengths, missing values, distribution by centre
4. Read a sample of responses with no theme in mind, just to see what's there

In [2]:
# 1. Load the raw data
import pandas as pd

df = pd.read_csv("../data/raw/City_Cultural_Centers_Audit_Community_Survey_-_Open_Response_Data_20260912.csv")

# Quick check that it loaded as expected
print(df.shape)
df.head()

(862, 7)


,Facility,Survey Item,Response,Auditor-assigned Category,Re-assigned response?,Translated?,Original Language
0,African American Cultural and Heritage Facility,Please let us know if you have any additional ...,TJ Ownes should be compensated on the level of...,NaN,False,False,English
1,African American Cultural and Heritage Facility,Please let us know if you have any additional ...,I would like to see more professional producti...,NaN,False,False,English
2,African American Cultural and Heritage Facility,Please let us know if you have any additional ...,"The dance studio floor is consistently dirty, ...",Negative,False,False,English
3,African American Cultural and Heritage Facility,Please let us know if you have any additional ...,This facility needs some TLC. It needs renovat...,Negative,False,False,English
4,African American Cultural and Heritage Facility,Please let us know if you have any additional ...,The facility needs to be updated and a deep cl...,Negative,False,False,English


In [4]:
# 2. Filter to the facilities-feedback responses only
# Survey Item is a free-text field, so I filter on it containing "facilities"
facilities_df = df[df["Survey Item"].str.contains("facilities", case=False)].copy()

# Reset the index so rows are numbered cleanly from 0
facilities_df = facilities_df.reset_index(drop=True)

print(f"Facilities responses: {len(facilities_df)}")
facilities_df["Facility"].value_counts()

Facilities responses: 204


Facility
Emma S. Barrientos Mexican American Cultural Center    92
Asian American Resource Center                         55
George Washington Carver Museum                        42
African American Cultural and Heritage Facility        15
Name: count, dtype: int64

In [6]:
# 3. Check response length and missing values
# Response length matters because very short responses ("N/A", "Good") need a
# different handling rule to long, multi-theme responses

facilities_df["response_length"] = facilities_df["Response"].str.len()

print(facilities_df["response_length"].describe())
print()
print("Missing values per column:")
print(facilities_df.isna().sum())

count     204.000000
mean      164.838235
std       191.813688
min         7.000000
25%        58.750000
50%       110.500000
75%       190.750000
max      1681.000000
Name: response_length, dtype: float64

Missing values per column:
Facility                      0
Survey Item                   0
Response                      0
Auditor-assigned Category    18
Re-assigned response?         0
Translated?                   0
Original Language             0
response_length               0
dtype: int64


## Reading a sample cold

Before building any codes, I want to read a genuine cross-section of responses
without trying to categorise them yet. I'm sampling across all four facilities
so I don't anchor on one centre's issues.

I'm not writing down themes at this stage. Just reading.

In [11]:
# 4. Sample a spread of responses to read manually
# Using a list comprehension instead of groupby().apply() to avoid a pandas
# version issue where the grouping column gets dropped from the result

sample = pd.concat([
    group.sample(min(10, len(group)), random_state=42)
    for _, group in facilities_df.groupby("Facility")
])

# Print each response in full, with its facility, so I can read them properly
# rather than have pandas truncate the text
for i, row in sample.iterrows():
    print(f"--- {row['Facility']} ---")
    print(row["Response"])
    print()

--- African American Cultural and Heritage Facility ---
Current room size limits gatherings to small groups. However, the location is an excellent central Austin choice.

--- African American Cultural and Heritage Facility ---
facilities are poorly designed. space should be upgraded

--- African American Cultural and Heritage Facility ---
TJ Ownes should be compensated on the level of all other staff that heads and manage other Cultural facilities on the behalf of the city of Austin.

--- African American Cultural and Heritage Facility ---
This "cultural center" is built on stolen land. Eminent domain is theft and violence against the Black community. Give it back.

--- African American Cultural and Heritage Facility ---
We need statues of pharoahs and black queens because we were also descendents of Kings and queens of Africa and America.  It needs to showcase black Indians in America as well.

--- African American Cultural and Heritage Facility ---
As a sixth generation Austinite, I 

## Notes from reading

**Recurring topics I'm seeing:**

- **Space size / capacity.** By far the most common thread. "Too small," "would
  benefit from an expansion," "rooms seem a bit small and can get crowded." Comes
  up across all four facilities.
- **Parking.** A specific, repeated complaint, especially at AARC and MACC.
  People describe it in real detail (parking tickets, overflow lots, "poor
  planning on the city's part").
- **Cleanliness / maintenance / condition.** Torn seating, dirty rooms, an
  outdated projector, a floor that "does not appear to have EVER been mopped."
  Distinct from the size complaints, this is about upkeep of what already exists.
- **Comparison between facilities, tied to equity.** Several AACHF and Carver
  responses explicitly compare their facility's size or funding to the MACC or
  AARC, and frame it as the Black community being under-resourced. This is not
  just "facility is too small," it's a specific claim about unequal treatment.
- **Accessibility and location.** A few responses raise physical access
  (wheelchair access at MACC) or geographic access (facility being far from the
  community it's meant to serve, e.g. "something east of I-35 would be more
  logical").
- **General satisfaction, no complaint.** A meaningful number of responses are
  short and simply positive ("All is good," "Well run with good facilities"),
  with no actionable content at all.
- **Off-topic or non-substantive.** A couple of responses aren't really about
  facilities at all (one is entirely about staff compensation, one is a general
  statement about land history rather than a facility comment).

**Things I want to be careful about:**

- **Multi-theme responses are common, not rare.** The long AACHF response about
  the dance studio raises cleanliness, size, comparison to other facilities, and
  an equity complaint, all in one response. My codebook needs to allow multiple
  codes per response, not force one label each.
- **The equity/comparison theme is easy to miss if I code too literally.**
  Several responses read like a facilities complaint on the surface but are
  really making a comparative claim about resource allocation. I want a
  specific code for this rather than folding it into "space size."
- **Positive responses need their own code, not just an absence of complaint.**
  Otherwise I'd have no way to report what people are happy about, only what
  they're unhappy about.
- **Short, low-content responses ("All is good," "stick to the fundamentals")
  need a clear rule.** They're not blank, but they're not very codeable either.
  I'll need to decide whether these count as a real "Positive, no detail" code
  or get excluded from thematic analysis while still counting in a satisfaction
  tally.

This gives me roughly six to seven candidate themes to test against the full
204 responses in Notebook 2: Space and Capacity, Maintenance and Condition,
Parking, Equity and Comparison, Accessibility and Location, General
Satisfaction (no specific issue), and Off-topic/Non-substantive.

## Testing the candidate themes against a larger sample

Before writing a formal codebook, I want to check whether the seven themes from
my first read hold up against more of the data, or whether they need splitting,
merging, or dropping. I'll pull a larger sample, read it against my seven
candidate themes, and note anywhere a response doesn't fit cleanly.

Candidate themes so far:
1. Space and Capacity
2. Maintenance and Condition
3. Parking
4. Equity and Comparison
5. Accessibility and Location
6. General Satisfaction (no specific issue)
7. Off-topic / Non-substantive

In [15]:
# 5. Pull a larger, different sample to test the candidate themes
# Excluding rows already seen in the first sample so this is genuinely new material

seen_indices = sample.index
remaining_df = facilities_df.drop(index=seen_indices)

test_sample = pd.concat([
    group.sample(min(15, len(group)), random_state=7)
    for _, group in remaining_df.groupby("Facility")
])

for i, row in test_sample.iterrows():
    print(f"[{i}] --- {row['Facility']} ---")
    print(row["Response"])
    print()

[3] --- African American Cultural and Heritage Facility ---
This facility needs some TLC. It needs renovation, expansion to accommodate more community programming or restructuring

[10] --- African American Cultural and Heritage Facility ---
Hours to use the facility meeting space is limited to day time use only. Not great for use by the community usage after working hours of the general public.

[7] --- African American Cultural and Heritage Facility ---
Austin is the worst mixed culture. Fuck this place

[6] --- African American Cultural and Heritage Facility ---
This facility needs an update to its video projector.  The last time I was there I helped jerry-rig and old projector and they said they always needed to do that to project to the screen in their auditorium.  City of Austin should really spend the money to update their projector and properly set it up.  However, I saw how they make events work with what they have I commend them for that.  Come on Austin, TX, spend more money